# common/training.ipynb — Huấn luyện, lưu mô hình và tổng hợp kết quả

Notebook chứa hàm dùng chung, không tự train và không tự ghi kết quả. Các notebook chính nạp tự động bằng `%run`. Khi sửa hàm, khởi động lại kernel rồi Run All notebook chính.


## Nạp các hàm phụ thuộc
Cell này cũng cho phép mở và Run All riêng notebook dùng chung.

In [ ]:
from pathlib import Path
import os
here = Path.cwd().resolve()
root = next((p for p in [here, *here.parents] if (p / 'common/data_io.ipynb').is_file()), None)
if root is None:
    raise FileNotFoundError('Mở notebook trong thư mục project đầy đủ.')
os.chdir(root)
get_ipython().run_line_magic('run', 'common/data_io.ipynb')
get_ipython().run_line_magic('run', 'common/splits.ipynb')
get_ipython().run_line_magic('run', 'common/metrics.ipynb')
get_ipython().run_line_magic('run', 'common/models.ipynb')


## Thư viện và hằng số

In [ ]:
import hashlib

import importlib.metadata

import json

from datetime import datetime, timezone

from pathlib import Path

from uuid import uuid4

import cloudpickle

import platform

import numpy as np

import pandas as pd

from sklearn.model_selection import GridSearchCV


## `fit_model`
Chọn tham số bằng 5 fold Train và fit lại trên toàn Train.

In [ ]:
def fit_model(name, df, split):
    train_idx = split['train']
    X, y = df.drop(columns=TARGET), df[TARGET]
    model, grid = build_model(name, X.iloc[train_idx], split['seed'])
    info = {'model': name, 'seed': split['seed'], 'best_params': {}, 'cv_macro_f1': None}
    if grid:
        positions = {row: i for i, row in enumerate(train_idx)}
        folds = [(np.array([positions[x] for x in f['train']]),
                  np.array([positions[x] for x in f['validation']])) for f in split['folds']]
        search = GridSearchCV(model, grid, scoring='f1_macro', cv=folds,
                              n_jobs=1, refit=True, error_score='raise')
        search.fit(X.iloc[train_idx], y.iloc[train_idx])
        model = search.best_estimator_
        info.update(best_params=search.best_params_, cv_macro_f1=float(search.best_score_))
    else:
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
    return model, info


## `save_predictions`
Lưu nhãn thật, nhãn dự đoán và xác suất theo đúng thứ tự lớp.

In [ ]:
def save_predictions(model, X, y, indices, output):
    pred = model.predict(X.iloc[indices])
    proba = model.predict_proba(X.iloc[indices])
    result = pd.DataFrame({'row_index': indices, 'actual': y.iloc[indices].to_numpy(), 'predicted': pred})
    for label in LABELS:
        col = list(model.classes_).index(label) if label in model.classes_ else None
        result[f'prob_{label}'] = proba[:, col] if col is not None else 0.0
    result.to_csv(output, index=False)
    return evaluate(y.iloc[indices], pred)


## `run_validation`
Mỗi lần chạy tạo thư mục artifacts riêng với model, metrics, cấu hình và dự đoán.

In [ ]:
def run_validation(name, data_path, split_path, artifacts_dir):
    df = load_data(data_path)
    split = load_splits(split_path, data_path, df)
    model, config = fit_model(name, df, split)
    run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid4().hex[:8]
    out = Path(artifacts_dir) / name / run_id
    out.mkdir(parents=True, exist_ok=False)
    estimator = model.named_steps['model'] if hasattr(model, 'named_steps') else model
    config.update(run_id=run_id, dataset_sha256=split['sha256'], model_settings=estimator.get_params(),
                  split_file_sha256=hashlib.sha256(Path(split_path).read_bytes()).hexdigest())
    metrics = save_predictions(model, df.drop(columns=TARGET), df[TARGET], split['validation'], out/'predictions.csv')
    save_model(model, out/'model.pkl')
    save_json(out/'metrics.json', metrics)
    save_json(out/'config.json', config)
    versions = {'python': platform.python_version()}
    for package in ('numpy', 'pandas', 'scikit-learn', 'joblib', 'matplotlib', 'torch', 'cloudpickle', 'ipython', 'nbformat', 'ipykernel'):
        try:
            versions[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            continue
    save_json(out/'environment.json', versions)
    return model, metrics, out


## `compare_validation`
Chỉ ghép các lượt chạy được chỉ định và có cùng bộ chia.

In [ ]:
def compare_validation(run_dirs, split_path, output):
    """Compare explicitly chosen runs, never silently select the latest run."""
    if not run_dirs:
        raise ValueError('Choose validation run directories before comparing.')
    expected = hashlib.sha256(Path(split_path).read_bytes()).hexdigest()
    rows = []
    for folder in map(Path, run_dirs):
        config = json.loads((folder/'config.json').read_text(encoding='utf-8'))
        if config['split_file_sha256'] != expected:
            raise ValueError(f'Run uses a different split: {folder}')
        metrics = json.loads((folder/'metrics.json').read_text(encoding='utf-8'))
        rows.append({'model': config['model'], 'run_id': config['run_id'],
                     'cv_macro_f1': config['cv_macro_f1'],
                     **{k: metrics[k] for k in ('accuracy', 'macro_f1', 'dropout_f2')}})
    result = pd.DataFrame(rows).sort_values('macro_f1', ascending=False)
    Path(output).parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(output, index=False)
    return result


## `run_final_test`
Fit lại Train + Validation sau khi khóa lựa chọn, rồi đánh giá Test một lần.

In [ ]:
def run_final_test(selected_run, data_path, split_path, artifacts_dir, selection_locked=False):
    if not selection_locked:
        raise ValueError('The team must lock the model and configuration before testing.')
    out = Path(artifacts_dir) / 'final_test'
    if out.exists():
        raise FileExistsError('Final Test results already exist. Do not evaluate candidates repeatedly.')
    df = load_data(data_path)
    split = load_splits(split_path, data_path, df)
    config = json.loads((Path(selected_run)/'config.json').read_text(encoding='utf-8'))
    if config['split_file_sha256'] != hashlib.sha256(Path(split_path).read_bytes()).hexdigest():
        raise ValueError('The chosen validation run uses a different split.')
    X, y = df.drop(columns=TARGET), df[TARGET]
    rows = split['train'] + split['validation']
    model, _ = build_model(config['model'], X.iloc[rows], split['seed'])
    model.set_params(**config['best_params'])
    estimator = model.named_steps['model'] if hasattr(model, 'named_steps') else model
    estimator.set_params(**config['model_settings'])
    model.fit(X.iloc[rows], y.iloc[rows])
    out.mkdir(parents=True, exist_ok=False)
    metrics = save_predictions(model, X, y, split['test'], out/'predictions.csv')
    save_model(model, out/'model.pkl')
    save_json(out/'metrics.json', metrics)
    save_json(out/'locked_config.json', {'validation_config': config, 'refit': 'train + validation'})
    return metrics, out


## `save_model`
Lưu cả class được khai báo trong notebook để có thể nạp trong kernel mới.

In [ ]:
def save_model(model, path):
    with Path(path).open('wb') as stream:
        cloudpickle.dump(model, stream)


## `load_saved_model`
Mở lại mô hình đã lưu; dùng cùng phiên bản Python và thư viện của lượt train.

In [ ]:
def load_saved_model(path):
    with Path(path).open('rb') as stream:
        return cloudpickle.load(stream)
